# Assignment 02: Object Detection

**Available:** Aug 26, 2025 until Sep 4, 2025 11:59pm

## Tasks:

1. Object Detection- refer to slides 8/26/2025 class
2. https://www.kaggle.com/datasets/awsaf49/coco-2017-dataset
3. Run detection performance comparing (inference only):
    - Faster R-CNN
    - DETR
    - DINO
    - Grounding DINO (use text prompt at your discretion)
4. Grading Criteria:
    - Report
    - Code
    - Video
    - Insight


## Import Packages and Setup

In [5]:
## Import Libraries

# Set CUDA_VISIBLE_DEVICES to make both GPUs visible
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

import torch
import torch.nn as nn
import torchvision
import cv2
import matplotlib.pyplot as plt
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch import optim
from tqdm.notebook import tqdm
from torchinfo import summary
import einops
import PIL
import numpy as np
import pandas as pd
# Use a pipeline as a high-level helper
from transformers import pipeline

# Install einops for tensor manipulation
%pip install einops

# Authorize Huggingface account
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv('/mnt/Storage02/SoftwareDev/CAP_6411_Assignments/.env')

# Get Hugging Face token
hf_token = os.getenv('HUGGINGFACE_HUB_TOKEN') or os.getenv('HF_TOKEN')

if hf_token:
    print("Found Hugging Face token in environment variables")
    
    # Install huggingface_hub if not already installed
    %pip install huggingface_hub
    
    from huggingface_hub import login, whoami
    
    try:
        # Login to Hugging Face Hub
        login(token=hf_token)
        
        # Verify login by getting user info
        user_info = whoami()
        print(f"Successfully authenticated with Hugging Face!")
        print(f"Logged in as: {user_info['name']}")
        
        # Set the token as environment variable for other libraries
        os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token
        os.environ['HF_TOKEN'] = hf_token
        
    except Exception as e:
        print(f"Authentication failed: {e}")
        print("Will proceed without pre-trained models if needed")
        hf_token = None
else:
    print("No Hugging Face token found in .env file")
    print("Please add HUGGINGFACE_HUB_TOKEN=your_token_here to your .env file")
    hf_token = None

# Comprehensive GPU diagnostics
print("\n=== GPU Diagnostics ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs detected: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print("\n=== All Available GPUs ===")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}:")
        print(f"  Name: {props.name}")
        print(f"  Total Memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"  Multi-processor count: {props.multi_processor_count}")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print()

# Device selection with preference for cuda:1 (A6000) -> cuda:0 (4090) -> cpu
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    device = torch.device('cuda:1')  # This should now be your A6000!
    print(f"Using GPU 1: {torch.cuda.get_device_name(1)}")
elif torch.cuda.is_available():
    device = torch.device('cuda:0')
    print(f"Using GPU 0: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print("Using CPU")

print(f"Selected device: {device}")

# If no logs folder exists, create one
if not os.path.exists("logs"):
    os.makedirs("logs")

# If no checkpoints folder exists, create one
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")

# If no data folder exists, create one
if not os.path.exists("data"):
    os.makedirs("data")

Note: you may need to restart the kernel to use updated packages.
Found Hugging Face token in environment variables
Note: you may need to restart the kernel to use updated packages.
Found Hugging Face token in environment variables
Note: you may need to restart the kernel to use updated packages.Note: you may need to restart the kernel to use updated packages.



Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Successfully authenticated with Hugging Face!
Logged in as: malneyugnfl

=== GPU Diagnostics ===
PyTorch version: 2.8.0
CUDA available: False
CUDA version: None
Number of GPUs detected: 0
Using CPU
Selected device: cpu


## Import and Process Data

In [6]:
# Import Dataset and Process it

import json
from PIL import Image
from torch.utils.data import Dataset
import os

class CocoDataset(Dataset):
    """
    Custom PyTorch Dataset class for loading COCO 2017 data.
    
    This class handles loading images and their corresponding annotations from the COCO dataset,
    making it compatible with PyTorch DataLoaders and various object detection models.
    """
    
    def __init__(self, root_dir, split="val2017", annotation_type="instances", transform=None, return_coco=False):
        """
        Initialize the COCO dataset.
        
        Args:
            root_dir (str): Path to the COCO dataset root directory (should contain 'annotations', 'train2017', 'val2017', etc.)
            split (str): Dataset split - one of 'train2017', 'val2017', or 'test2017'
            annotation_type (str): Type of annotations - 'instances', 'captions', or 'person_keypoints'
            transform (callable, optional): Transform to apply to images and targets
            return_coco (bool): If True, also return the raw COCO annotation dict for each sample
        """
        # Store initialization parameters
        self.root_dir = root_dir
        self.split = split
        self.transform = transform
        self.return_coco = return_coco
        
        # Set up image directory path
        self.img_dir = os.path.join(root_dir, split)
        self.annotation_type = annotation_type
        
        # Handle test split (which has no annotations)
        if split == "test2017":
            self.annotations = None
            # For test set, just get list of image filenames
            self.imgs = sorted(os.listdir(self.img_dir))
        else:
            # Load annotation file for train/val splits
            ann_file = os.path.join(
                root_dir, "annotations", f"{annotation_type}_{split}.json"
            )
            
            # Load COCO annotations JSON file
            with open(ann_file, "r") as f:
                self.coco = json.load(f)
            
            # Extract images and annotations from COCO data
            self.imgs = self.coco["images"]  # List of image metadata
            self.anns = self.coco["annotations"]  # List of annotation objects
            
            # Create mapping from image_id to list of annotations for that image
            # This allows fast lookup of annotations for a given image
            self.imgid_to_anns = {}
            for ann in self.anns:
                img_id = ann["image_id"]
                if img_id not in self.imgid_to_anns:
                    self.imgid_to_anns[img_id] = []
                self.imgid_to_anns[img_id].append(ann)
            
            # Create mapping from image_id to filename for easy access
            self.imgid_to_filename = {img["id"]: img["file_name"] for img in self.imgs}
            
            # Create mapping from category_id to category name (useful for visualization)
            self.catid_to_name = {cat["id"]: cat["name"] for cat in self.coco.get("categories", [])}

    def __len__(self):
        """Return the total number of images in the dataset."""
        return len(self.imgs)

    def __getitem__(self, idx):
        """
        Get a single sample from the dataset.
        
        Args:
            idx (int): Index of the sample to retrieve
            
        Returns:
            tuple: (image, target) where:
                - image: PIL Image or transformed tensor
                - target: Dictionary containing bounding boxes, labels, and other metadata
        """
        
        # Handle test split (no annotations available)
        if self.split == "test2017":
            img_name = self.imgs[idx]
            img_path = os.path.join(self.img_dir, img_name)
            image = Image.open(img_path).convert("RGB")
            target = {}  # Empty target for test images
        else:
            # Get image metadata and construct path
            img_info = self.imgs[idx]
            img_id = img_info["id"]
            img_name = img_info["file_name"]
            img_path = os.path.join(self.img_dir, img_name)
            
            # Load image and convert to RGB
            image = Image.open(img_path).convert("RGB")
            
            # Get all annotations for this image
            anns = self.imgid_to_anns.get(img_id, [])
            
            # Extract bounding box and label information from annotations
            boxes = []      # Bounding boxes in [x, y, width, height] format
            labels = []     # Category IDs for each object
            areas = []      # Area of each bounding box
            iscrowd = []    # Whether each annotation represents a crowd of objects
            
            for ann in anns:
                # Only process annotations that have bounding boxes
                if "bbox" in ann:
                    boxes.append(ann["bbox"])  # COCO format: [x, y, width, height]
                    labels.append(ann["category_id"])
                    areas.append(ann.get("area", 0))
                    iscrowd.append(ann.get("iscrowd", 0))
            
            # Create target dictionary in PyTorch format
            # Convert lists to tensors, handling empty cases
            target = {
                "boxes": torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0,4), dtype=torch.float32),
                "labels": torch.tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,), dtype=torch.int64),
                "image_id": torch.tensor([img_id]),  # Unique identifier for this image
                "area": torch.tensor(areas, dtype=torch.float32) if areas else torch.zeros((0,), dtype=torch.float32),
                "iscrowd": torch.tensor(iscrowd, dtype=torch.int64) if iscrowd else torch.zeros((0,), dtype=torch.int64),
            }
        
        # Apply transforms if provided (e.g., normalization, resizing)
        if self.transform:
            image = self.transform(image)
        
        # Optionally return raw COCO annotations along with processed data
        if self.return_coco and self.split != "test2017":
            return image, target, anns
        
        return image, target

# Example usage:
# Create a dataset instance
# dataset = CocoDataset(
#     root_dir="data/coco2017",              # Path to COCO dataset
#     split="val2017",                       # Use validation split
#     annotation_type="instances",           # Object detection annotations
#     transform=transforms.ToTensor(),       # Convert PIL Images to tensors
# )
# 
# # Get a single sample
# img, target = dataset[0]
# print(f"Image shape: {img.shape}")
# print(f"Number of objects: {len(target['labels'])}")
# print(f"Object categories: {target['labels']}")
# print(f"Bounding boxes: {target['boxes']}")
#
# # Use with DataLoader for batching
# from torch.utils.data import DataLoader
# dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
# for batch_imgs, batch_targets in dataloader:
#     # Process batch
#     break

In [7]:
# Create Instance of dataset for inference

# Define transforms for inference (normalize to match model expectations)
inference_transform = transforms.Compose([
    transforms.ToTensor(),  # Convert PIL Image to tensor and scale to [0,1]
    # Note: Different models may need different normalization
    # We'll apply model-specific transforms during inference
])

# Create dataset instance pointing to the COCO 2017 validation set
coco_dataset = CocoDataset(
    root_dir="data/coco2017",           # Path to COCO dataset
    split="val2017",                    # Use validation split for inference
    annotation_type="instances",        # Object detection annotations
    transform=inference_transform,      # Apply transforms
    return_coco=False                   # Don't need raw COCO annotations for inference
)

print(f"Dataset loaded successfully!")
print(f"Number of images in validation set: {len(coco_dataset)}")
print(f"Available categories: {len(coco_dataset.catid_to_name)} classes")

# Define a proper collate function that can be pickled
def collate_fn(batch):
    """
    Custom collate function for object detection.
    Since we're using batch_size=1, this just returns the batch as-is.
    """
    return batch

# Create DataLoader for batched inference
# Using batch_size=1 for inference to avoid complications with variable-sized images
inference_dataloader = DataLoader(
    coco_dataset,
    batch_size=1,                       # Process one image at a time
    shuffle=False,                      # No need to shuffle for inference
    num_workers=0,                      # Set to 0 to avoid pickling issues with custom classes
    collate_fn=collate_fn,             # Use proper function instead of lambda
    pin_memory=True if torch.cuda.is_available() else False  # Speed up GPU transfer
)

print(f"DataLoader created with batch size: {inference_dataloader.batch_size}")
print(f"Number of batches: {len(inference_dataloader)}")

# Test loading a single sample to verify everything works
print("\n=== Testing dataset loading ===")
try:
    sample_image, sample_target = coco_dataset[0]
    print(f"Sample loaded successfully!")
    print(f"Image shape: {sample_image.shape}")
    print(f"Image dtype: {sample_image.dtype}")
    print(f"Number of objects in image: {len(sample_target['labels'])}")
    print(f"Object categories in image: {sample_target['labels'].tolist()}")
    print(f"Image ID: {sample_target['image_id'].item()}")
    
    # Show some category names if available
    if len(sample_target['labels']) > 0:
        print("Category names:")
        for label in sample_target['labels'][:5]:  # Show first 5 categories
            cat_name = coco_dataset.catid_to_name.get(label.item(), f"Unknown ({label.item()})")
            print(f"  - {cat_name}")
            
except Exception as e:
    print(f"Error loading sample: {e}")

# Test DataLoader iteration
print("\n=== Testing DataLoader ===")
try:
    # Get first batch
    first_batch = next(iter(inference_dataloader))
    image, target = first_batch[0]  # Extract from list since batch_size=1
    print(f"DataLoader working correctly!")
    print(f"Batch image shape: {image.shape}")
    print(f"Batch target keys: {list(target.keys())}")
    
except Exception as e:
    print(f"Error with DataLoader: {e}")

Dataset loaded successfully!
Number of images in validation set: 5000
Available categories: 80 classes
DataLoader created with batch size: 1
Number of batches: 5000

=== Testing dataset loading ===
Sample loaded successfully!
Image shape: torch.Size([3, 427, 640])
Image dtype: torch.float32
Number of objects in image: 19
Object categories in image: [44, 67, 1, 49, 51, 51, 79, 1, 47, 47, 51, 51, 56, 50, 56, 56, 79, 57, 81]
Image ID: 397133
Category names:
  - bottle
  - dining table
  - person
  - knife
  - bowl

=== Testing DataLoader ===
DataLoader working correctly!
Batch image shape: torch.Size([3, 427, 640])
Batch target keys: ['boxes', 'labels', 'image_id', 'area', 'iscrowd']


## Import Faster R-CNN

In [ ]:
# Load Faster R-CNN model
from torchvision.models import detection
import torchvision.transforms as T

# Load pre-trained Faster R-CNN model with ResNet-50 backbone
faster_rcnn = detection.fasterrcnn_resnet50_fpn(pretrained=True)
faster_rcnn.eval()  # Set to evaluation mode
faster_rcnn = faster_rcnn.to(device)


## Import DinoV3 Model

In [ ]:
# Load model directly
# Documentation: https://huggingface.co/facebook/dinov3-vit7b16-pretrain-lvd1689m
from transformers import AutoImageProcessor, AutoModel

processor = AutoImageProcessor.from_pretrained("facebook/dinov3-vit7b16-pretrain-lvd1689m")
model = AutoModel.from_pretrained("facebook/dinov3-vit7b16-pretrain-lvd1689m")

## Import DETR Model

In [ ]:
# Load model directly
# Documentation: https://huggingface.co/facebook/detr-resnet-50
from transformers import AutoImageProcessor, AutoModelForObjectDetection

processor = AutoImageProcessor.from_pretrained("facebook/detr-resnet-50")
model = AutoModelForObjectDetection.from_pretrained("facebook/detr-resnet-50")

## Import Grounding-DINO Model

In [ ]:
# Load model directly
# Documentation: https://huggingface.co/IDEA-Research/grounding-dino-base
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

processor = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-base")
model = AutoModelForZeroShotObjectDetection.from_pretrained("IDEA-Research/grounding-dino-base")